# Summary: Clustering Method Comparison Across All Datasets

Aggregates results from `01_wine.ipynb`, `02_mall_customers.ipynb`,
`03_fashion_mnist.ipynb`, and `04_make_moons.ipynb` (each writes its metrics
to `results/<dataset>_results.csv`).

**Run the other 4 notebooks first** so `results/*.csv` exist before running
this one.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt

from sc4020 import generate_figures as gf
from sc4020.paths import results_dir

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 50)

RESULTS_DIR = results_dir("datasets")
NOTEBOOKS = ["wine_tuned_methods", "mall_customers", "fashion_mnist", "make_moons"]

In [ ]:
frames = []
for name in NOTEBOOKS:
    path = RESULTS_DIR / f"{name}_results.csv"
    if not path.exists():
        print(f"missing {path} - run notebooks/{name}.ipynb first")
        continue
    frames.append(pd.read_csv(path))
all_results = pd.concat(frames, ignore_index=True)
all_results.head()

## Combined metrics table

One row per (dataset, method). Mall Customers has no external metrics
(ARI/NMI/AMI/FMI are `NaN`) since it has no ground truth.

In [ ]:
metric_cols = [
    "dataset", "method", "fit_seconds", "n_clusters", "noise_fraction", "largest_cluster_share",
    "silhouette", "davies_bouldin", "calinski_harabasz", "ari", "nmi", "ami", "fmi",
]
display(all_results[metric_cols].sort_values(["dataset", "method"]))

## Metric comparison charts

In [ ]:
for metric in ["silhouette", "davies_bouldin", "calinski_harabasz"]:
    gf.plot_metrics_bar(all_results, metric)
    plt.show()

In [ ]:
external_df = all_results.dropna(subset=["ari"])
for metric in ["ari", "nmi", "ami", "fmi"]:
    gf.plot_metrics_bar(external_df, metric)
    plt.show()

## Runtime comparison (proposed metric)

`fit_seconds` — wall-clock fit time — is reported as a first-class metric
throughout. It captures a dimension the required metrics don't: how each
method's cost scales with `n`/dimensionality, which is directly why
Fashion-MNIST needed subsampling for Spectral/DBSCAN/HDBSCAN in the first
place.

In [ ]:
gf.plot_metrics_bar(all_results, "fit_seconds", log_scale=True)
plt.show()

## Findings

- **No single method wins everywhere.** K-Means++ and GMM are fast and strong
  when clusters are roughly convex/Gaussian and K is known (Wine), but fail
  badly on non-convex structure (make_moons).
- **DBSCAN and HDBSCAN win decisively on non-convex, well-separated data**
  (make_moons: ARI/NMI near 1.0) but are noticeably weaker in higher-dimensional
  or noisier feature spaces (Wine's 13-D space, Fashion-MNIST's 50-D PCA space),
  where a single global density threshold struggles — visible in their elevated
  `noise_fraction` and mismatched `n_clusters`.
- **Spectral Clustering** is the most consistently competitive method across
  shapes (convex and non-convex alike) when given an appropriate affinity
  (`rbf` for globular data, `nearest_neighbors` for manifold-shaped data), at
  the cost of being the slowest method (`fit_seconds`) and requiring K upfront.
- **Mall Customers** (no ground truth) is a reminder that internal metrics
  alone can't confirm a clustering is "correct" — only that it's internally
  well-separated; different methods here favor different segment counts/shapes
  with no way to declare a winner without domain judgment.
- **Proposed metrics beyond the required set** — Adjusted Mutual Information
  (chance-corrected, robust when predicted vs. true cluster counts differ, as
  with DBSCAN/HDBSCAN), Fowlkes-Mallows Index (pairwise precision/recall,
  complements ARI), noise ratio / clusters-found (diagnostic context for
  density-based methods), and fit time (practical scalability) — all added
  signal that Silhouette/DB/CH/ARI/NMI alone didn't capture in this comparison.